# 01b Prepare remaining folds 2-5

Use raw PPG-DaLiA, not the prepared fold 1 dataset. CPU is sufficient.
Keep the verified label alignment and preprocessing protocol from notebook 01.
Each fold fits normalization using its own train subjects only.
Outputs: artifacts/prepared/fold_02..fold_05 and artifacts/ppg-cvae-prepared-folds02-05.zip.
This stage creates split arrays; it does not evaluate models on test.


In [1]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLDS = (2, 3, 4, 5)
SEED = 42
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Configure DATA_ROOT to the mounted PPG-DaLiA dataset: {DATA_ROOT}')


=== CHECKING /kaggle/input ===
Contents of /kaggle/input (1 items): ['datasets']
  Folder datasets contains 1 items
Auto-discovered DATA_ROOT via subject files: /kaggle/input/datasets
{'DATA_ROOT': '/kaggle/input/datasets', 'OUTPUT_ROOT': 'artifacts', 'device': 'cpu', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'torch': '2.11.0+cpu'}


In [2]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
for fold in FOLDS:
    print(get_fold(fold))


FoldSubjects(fold_id=2, train_subjects=frozenset({'S12', 'S11', 'S1', 'S10', 'S2', 'S14', 'S3', 'S13', 'S15'}), val_subjects=frozenset({'S7', 'S8', 'S9'}), test_subjects=frozenset({'S6', 'S4', 'S5'}))
FoldSubjects(fold_id=3, train_subjects=frozenset({'S5', 'S6', 'S1', 'S2', 'S4', 'S14', 'S3', 'S13', 'S15'}), val_subjects=frozenset({'S12', 'S10', 'S11'}), test_subjects=frozenset({'S7', 'S8', 'S9'}))
FoldSubjects(fold_id=4, train_subjects=frozenset({'S5', 'S6', 'S8', 'S1', 'S9', 'S2', 'S4', 'S7', 'S3'}), val_subjects=frozenset({'S14', 'S13', 'S15'}), test_subjects=frozenset({'S12', 'S10', 'S11'}))
FoldSubjects(fold_id=5, train_subjects=frozenset({'S12', 'S11', 'S5', 'S6', 'S8', 'S10', 'S9', 'S7', 'S4'}), val_subjects=frozenset({'S2', 'S3', 'S1'}), test_subjects=frozenset({'S14', 'S13', 'S15'}))


In [3]:
from ppg_cvae.data.prepare import prepare_fold
# Synchronized SX.pkl: label[j] aligns with BVP[128*j : 128*j+512].
# Official readme specifies 8 s / 2 s windows; s0=0 is supported by
# the unique common offset compatible with the audited S1..S15 counts.
# See docs/data_sources/PPG_DaLiA_alignment.md for evidence and scope.
LABEL_START_SAMPLE = 0
ALIGNMENT_SOURCE = ('UCI PPG-DaLiA official readme, sections I.2 and III.3, pp. 1 and 3-4; '
                    'https://archive.ics.uci.edu/ml/machine-learning-databases/00495/readme.pdf; '
                    'synchronized SX.pkl, 8-second windows shifted by 2 seconds; '
                    's0=0 inferred from segmentation convention and uniquely supported by all 15 '
                    'subject counts in Kaggle ameersifat53/ppg-dalia-dataset audit job '
                    '20261003-171957-ff75c8, checked 2026-10-04.')
if LABEL_START_SAMPLE is None or not ALIGNMENT_SOURCE:
    raise ValueError('Set LABEL_START_SAMPLE and ALIGNMENT_SOURCE after verifying label alignment in the dataset readme.')
import json
import zipfile
import pandas as pd
summaries = []
for fold in FOLDS:
    print(f'Preparing fold {fold}', flush=True)
    prepared_dir = OUTPUT_ROOT/'prepared'/f'fold_{fold:02d}'
    manifest = prepare_fold(DATA_ROOT, prepared_dir, fold,
                            label_start_sample=LABEL_START_SAMPLE,
                            alignment_source=ALIGNMENT_SOURCE)
    print(manifest.groupby('split').valid.agg(['size','sum']), flush=True)
    for split, group in manifest.groupby('split'):
        summaries.append(dict(fold=fold, split=split, windows=len(group),
                              valid_windows=int(group.valid.sum()),
                              subjects=group.subject_id.nunique()))
    # Release each fold's manifest before preparing the next one.
    del manifest
pd.DataFrame(summaries).to_csv(OUTPUT_ROOT/'prepared_remaining_folds_summary.csv', index=False)
archive_path = OUTPUT_ROOT/'ppg-cvae-prepared-folds02-05.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for fold in FOLDS:
        for path in sorted((OUTPUT_ROOT/'prepared'/f'fold_{fold:02d}').iterdir()):
            if path.is_file():
                archive.write(path, path.relative_to(OUTPUT_ROOT).as_posix())
print('Prepared folds:', FOLDS, flush=True)
print('Dataset upload archive:', archive_path, 'bytes:', archive_path.stat().st_size, flush=True)


Preparing fold 2


        size    sum
split              
test   11843  11837
train  39872  39854
val    12982  12976


Preparing fold 3


        size    sum
split              
test   12982  12976
train  37919  37901
val    13796  13790


Preparing fold 4


        size    sum
split              
test   13796  13790
train  37894  37876
val    13007  13001


Preparing fold 5


        size    sum
split              
test   13007  13001
train  38621  38603
val    13069  13063


Prepared folds: (2, 3, 4, 5)


Dataset upload archive: artifacts/ppg-cvae-prepared-folds02-05.zip bytes: 130099935
